# Jukebox 1B Lyrics Quickstart

This notebook is tuned for your fork and focuses on the lightest useful Colab flow: load `1b_lyrics`, generate one short top-level sample from text, and optionally continue from your own uploaded audio.

Important: this notebook only generates the top level, so the output will sound noisy compared with a fully upsampled Jukebox sample. The advantage is that it is much faster and more realistic for Colab experimentation.


## Step 1: Install From The Fork

This pulls the notebook code directly from `emcee3/jukebox`, so anyone opening your fork gets your modified version instead of the archived upstream setup.


In [0]:
REPO_URL = "https://github.com/emcee3/jukebox.git"
!pip install git+{REPO_URL}

## Step 2: Confirm The GPU Runtime

This quickstart expects a Colab GPU runtime. If `nvidia-smi` fails here, switch the notebook runtime to GPU before continuing.


In [0]:
!nvidia-smi

## Step 3: Import Helpers And Set Up The Device

The custom `setup_dist_from_mpi()` path in your fork now falls back cleanly to a normal single-GPU Colab session instead of assuming MPI is present.


In [0]:
import torch as t
from IPython.display import Audio
from google.colab import files
from jukebox.make_models import make_prior, make_vqvae, MODELS
from jukebox.hparams import Hyperparams, setup_hparams
from jukebox.sample import _sample, load_prompts
from jukebox.utils.dist_utils import setup_dist_from_mpi

rank, local_rank, device = setup_dist_from_mpi()
device

## Step 4: Load The Smallest Practical Lyrics Model

`1b_lyrics` is the most realistic Jukebox model to try in Colab.

Two timing values matter here:
- `sample_length`: how much top-level audio we actually generate
- `total_length`: the song-level conditioning value the model expects

For `1b_lyrics`, the minimum valid sampled window is about `17.83s`, while the minimum valid song-level `total_length` label is slightly larger at about `17.84s`.


In [0]:
model = "1b_lyrics"
hps = Hyperparams()
hps.sr = 44100
hps.n_samples = 1
hps.levels = 3
hps.hop_fraction = [0.5, 0.5, 0.125]
sample_window_length = 786432
minimum_total_length = 786816

vqvae_name, *prior_names = MODELS[model]
vqvae_hps = setup_hparams(vqvae_name, dict(sample_length=sample_window_length))
vqvae = make_vqvae(vqvae_hps, device)
top_prior = make_prior(setup_hparams(prior_names[-1], dict()), vqvae, device)
hps.sample_length = sample_window_length
hps.total_length = minimum_total_length
dict(sample_seconds=hps.sample_length / hps.sr, total_seconds=hps.total_length / hps.sr, raw_to_tokens=top_prior.raw_to_tokens)

## Step 5: Choose The Musical Conditioning

These controls affect both flows below:
- artist
- genre
- lyrics

Even when you upload your own sample later, the continuation is still influenced by these labels. Try editing them to explore how strongly the model bends the continuation toward different styles or lyrical content.


In [0]:
metas = [dict(
    artist="Zac Brown Band",
    genre="Country",
    total_length=hps.total_length,
    offset=0,
    lyrics="""I met a traveller from an antique land,
    Who said Two vast and trunkless legs of stone
    Stand in the desert. Near them, on the sand,
    Half sunk a shattered visage lies, whose frown,
    And wrinkled lip, and sneer of cold command,
    Tell that its sculptor well those passions read
    Which yet survive, stamped on these lifeless things,
    The hand that mocked them, and the heart that fed;
    And on the pedestal, these words appear:
    My name is Ozymandias, King of Kings;
    Look on my Works, ye Mighty, and despair!
    Nothing beside remains. Round the decay
    Of that colossal Wreck, boundless and bare
    The lone and level sands stretch far away
    """
)] * hps.n_samples
labels = [None, None, top_prior.labeller.get_batch_labels(metas, 'cuda')]
labels[-1]['y'].shape

## Step 6: Define Small Reusable Helpers

The notebook uses two generation modes:
- text-only generation from scratch
- continuation from your own uploaded audio

These helpers keep both flows consistent while still making the editable knobs easy to find.


In [0]:
def make_labels():
    return [None, None, top_prior.labeller.get_batch_labels(metas, 'cuda')]

def top_level_sampling_kwargs(temp=0.98):
    return [
        dict(temp=0.99, fp16=True, max_batch_size=16, chunk_size=32),
        dict(temp=0.99, fp16=True, max_batch_size=16, chunk_size=32),
        dict(temp=temp, fp16=True, max_batch_size=16, chunk_size=32),
    ]

def run_top_level(zs, output_name, temp=0.98):
    hps.name = output_name
    return _sample(zs, make_labels(), top_level_sampling_kwargs(temp), [None, None, top_prior], [2], hps)


## Step 7: Generate A Short Sample From Scratch

This is the cleanest first test. If this works, the model, labels, and top-level sampling path are all functioning.

Try changing `sampling_temperature` if you want more or less randomness.


In [0]:
sampling_temperature = 0.98
text_zs = [t.zeros(hps.n_samples, 0, dtype=t.long, device='cuda') for _ in range(len(prior_names))]
text_zs = run_top_level(text_zs, output_name='text_samples', temp=sampling_temperature)

In [0]:
Audio('text_samples/level_2/item_0.wav')

## Step 8: Upload Your Own Audio Prompt

Use this when you want Jukebox to continue from your own audio rather than starting from scratch.

Good prompt candidates:
- short WAV clips
- simple melodic fragments
- voice memos or instrument riffs

Jukebox does not behave like a precise editor. It is better to think of this as *continue or reinterpret my sample* rather than *apply a small edit to my exact waveform*.


In [0]:
uploaded = files.upload()
uploaded_audio_path = next(iter(uploaded)) if uploaded else None
uploaded_audio_path

In [0]:
assert uploaded_audio_path is not None, 'Upload an audio file first.'
Audio(uploaded_audio_path)

## Step 9: Continue From Your Uploaded Audio

Choose how much of your clip to use as the prompt. The prompt must be shorter than the full sampled window, so values around `4` to `12` seconds are a good place to start.

This path encodes your prompt into Jukebox tokens first, then asks the top prior to continue from there.


In [0]:
prompt_length_in_seconds = 8
sampling_temperature = 0.98
assert uploaded_audio_path is not None, 'Upload an audio file first.'
prompt_duration = (int(prompt_length_in_seconds * hps.sr) // top_prior.raw_to_tokens) * top_prior.raw_to_tokens
assert 0 < prompt_duration < hps.sample_length, 'Choose a prompt shorter than the full sample window.'
prompt_audio = load_prompts([uploaded_audio_path], prompt_duration, hps)
prompted_zs = top_prior.encode(prompt_audio, start_level=0, end_level=len(prior_names), bs_chunks=prompt_audio.shape[0])
prompted_zs = run_top_level(prompted_zs, output_name='prompted_samples', temp=sampling_temperature)
dict(prompt_seconds=prompt_duration / hps.sr, generated_seconds=hps.sample_length / hps.sr)

In [0]:
Audio('prompted_samples/level_2/item_0.wav')

## Things To Experiment With

Once the notebook works end to end, the easiest knobs to play with are:
- `artist`
- `genre`
- `lyrics`
- `sampling_temperature`
- `prompt_length_in_seconds`

Changing these will not make Jukebox precise or predictable, but they are the main levers you can use to steer the result.
